For licensing see accompanying LICENSE file.  
Copyright (C) 2025 Apple Inc. All Rights Reserved.

# Semantic Regex LLM Feature Descriptions
This notebook shows example usage of how to generate and evaluate a semantic regex.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

import methods
import features


In [19]:
# Helper functions
def print_result(result):
    print('_' * 65)
    print(f"{result['description']['method'].upper()}")
    print(f"FEATURE: {result['feature']}")
    print(f"DESCRIPTION: {result['description']['description']}")
    print("METRICS:")
    for metric, eval in result['evaluation'].items():
        print(f"{metric}: {eval['value']:.4f}")
    print("\n")

In [20]:
# Hyperparameters for generating and evaluating feature descriptions
description_model_name="gpt-4o-mini"
data_model_name="gpt-4o-mini"
eval_model_name="gpt-4o-mini"
n_data_examples=10
n_tokens_per_sample=32
show_breaks=True
ignore_first_token=True
logging=False
activation_threshold=0.3
sampling_method='top'
metric_list = ['clarity', 'responsiveness', 'purity', 'detection', 'fuzzing']
seed = 97
experiment_dir = './artifacts/experiments/'
experiment_hash = 'cf1daa'

In [21]:
# Choose a feature to describe
model_id = 'gpt2-small'
source = 'res-jb'
layer = f'{0}-{source}'
index = 0
feature = features.Feature(model_id, layer, index)

In [22]:
# Initialize the semantic regex method
semantic_regex_experiment_dir = os.path.join(experiment_dir, f'{experiment_hash}_semantic_regex_{model_id}_{source}')
semantic_regex = methods.SemanticRegex(
    output_dir=semantic_regex_experiment_dir,
    seed=seed,
    subject_model=None
)
semantic_regex_result = semantic_regex.generate_and_evaluate(
    feature=feature,
    model_name=description_model_name,
    eval_model_name=eval_model_name,
    data_model_name=data_model_name,
    n_data_examples=n_data_examples,
    n_tokens_per_sample=n_tokens_per_sample,
    sampling_method=sampling_method,
    show_breaks=show_breaks,
    logging=logging,
    ignore_first_token=ignore_first_token,
    metrics=metric_list,
    activation_threshold=activation_threshold
)
print_result(semantic_regex_result)

_________________________________________________________________
METHOD: semantic_regex
FEATURE: gpt2-small_0-res-jb_0

SEMANTIC_REGEX - Accessing results from disk for feature gpt2-small_0-res-jb_0
DESCRIPTION: @{:context military units:}([:symbol Brigade:])

SEMANTIC_REGEX - Accessing evaluation results from disk for feature gpt2-small_0-res-jb_0
_________________________________________________________________
SEMANTIC_REGEX
FEATURE: {'model_id': 'gpt2-small', 'layer': '0-res-jb', 'index': 0}
DESCRIPTION: @{:context military units:}([:symbol Brigade:])
METRICS:
clarity: 0.9800
responsiveness: 0.9754
purity: 0.9611
detection: 0.9000
fuzzing: 0.8833
faithfulness: 1.0000




In [25]:
# Compare to token-act-pair baseline
token_act_pair_experiment_dir = os.path.join(experiment_dir, f'{experiment_hash}_oai_token-act-pair_{model_id}_{source}')
token_act_pair_method = methods.OAITokenActPair(output_dir=token_act_pair_experiment_dir, seed=seed)
token_act_pair_result = token_act_pair_method.generate_and_evaluate(
    feature=feature,
    model_name=description_model_name,
    eval_model_name=eval_model_name,
    data_model_name=data_model_name,
    n_data_examples=n_data_examples,
    n_tokens_per_sample=n_tokens_per_sample,
    show_breaks=show_breaks,
    logging=logging,
    ignore_first_token=ignore_first_token,
    metrics=metric_list,
)
print_result(token_act_pair_result)

_________________________________________________________________
METHOD: oai_token-act-pair

FEATURE: gpt2-small_0-res-jb_0

OAI_TOKEN-ACT-PAIR - Accessing results from disk for feature gpt2-small_0-res-jb_0
DESCRIPTION: references to military units or brigades.

OAI_TOKEN-ACT-PAIR - Accessing evaluation results from disk for feature gpt2-small_0-res-jb_0
_________________________________________________________________
OAI_TOKEN-ACT-PAIR
FEATURE: {'model_id': 'gpt2-small', 'layer': '0-res-jb', 'index': 0}
DESCRIPTION: references to military units or brigades.
METRICS:
clarity: 0.4800
responsiveness: 0.9610
purity: 0.9329
detection: 0.9167
fuzzing: 0.9333




In [26]:
# Compare to max-acts baseline
max_acts_experiment_dir = os.path.join(experiment_dir, f'{experiment_hash}_eleuther_acts_top20_{model_id}_{source}')
max_acts_method = methods.OAITokenActPair(output_dir=max_acts_experiment_dir, seed=seed)
max_acts_result = max_acts_method.generate_and_evaluate(
    feature=feature,
    model_name=description_model_name,
    eval_model_name=eval_model_name,
    data_model_name=data_model_name,
    n_data_examples=n_data_examples,
    n_tokens_per_sample=n_tokens_per_sample,
    show_breaks=show_breaks,
    logging=logging,
    ignore_first_token=ignore_first_token,
    metrics=metric_list,
)
print_result(max_acts_result)

_________________________________________________________________
METHOD: oai_token-act-pair

FEATURE: gpt2-small_0-res-jb_0

OAI_TOKEN-ACT-PAIR - Accessing results from disk for feature gpt2-small_0-res-jb_0
DESCRIPTION: Frequent references to military units and formations, specifically "Brigade," indicating a focus on organized groups within a military context.

OAI_TOKEN-ACT-PAIR - Accessing evaluation results from disk for feature gpt2-small_0-res-jb_0
_________________________________________________________________
ELEUTHER_ACTS_TOP20
FEATURE: {'model_id': 'gpt2-small', 'layer': '0-res-jb', 'index': 0}
DESCRIPTION: Frequent references to military units and formations, specifically "Brigade," indicating a focus on organized groups within a military context.
METRICS:
clarity: 1.0000
responsiveness: 0.9617
purity: 0.9118
detection: 0.9500
fuzzing: 0.9500


